In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

# load the raw data
df = pd.read_csv("data/raw/support2.csv")
print("Loaded:", df.shape)

# check for dupes before we do anything to the data
raw_dupes = df.duplicated().sum()
print("Duplicate rows in raw data:", raw_dupes)

df = df.rename(columns={'d.time': 'd_time', 'num.co': 'num_co'})
TARGET = 'hospdead'

leakage_cols = ['death', 'd_time', 'slos', 'surv2m', 'surv6m', 'prg2m', 'prg6m', 'sfdm2', 'hday']
drop_cols = [c for c in leakage_cols if c in df.columns]
df_model = df.drop(columns=drop_cols)

# tracking shape at each step
shape_log = {}
shape_log['raw'] = df.shape
shape_log['after_leakage_drop'] = df_model.shape

print(f"Dropped {len(drop_cols)} leakage/administrative columns: {drop_cols}")
print(f"Remaining shape: {df_model.shape}")



In [ ]:
# quick sanity check on the categorical columns (look for typos, weird captials etc). 
cat_check_cols = df_model.select_dtypes(include='object').columns.tolist()

print("Unique values per categorical column (excluding NaN):\n")
for c in cat_check_cols:
    vals = df_model[c].dropna().unique()
    print(f"{c} ({len(vals)} unique): {sorted(vals.tolist())[:10]}{' ...' if len(vals) > 10 else ''}")

print("\nAge range: min =", df_model['age'].min(), " max =", df_model['age'].max())
# looks fine, nothing weird here

# these can't actually be 0 or negative in real life & turn them into NaN so they get handled properly
for c in ['meanbp', 'hrt', 'resp']:
    df_model.loc[df_model[c] == 0, c] = pd.NA

for c in ['totmcst', 'dnrday']:
    df_model.loc[df_model[c] < 0, c] = pd.NA

